In [2]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 50)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

In [3]:
# Method 1: full_call_tracker_all - Monthly, All BMS, lcdf-based, since 2025-05-01
# Groups by: pricing_hurdle_name, year, month
q1 = """
SELECT pricing_hurdle_name,
       DATE_PART('year', rh.reqappdate) AS year,
       DATE_PART('month', rh.reqappdate) AS month,
       MIN(rh.reqappdate) AS first_date,
       COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END) AS apps,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) AS cons,
       SUM(booked) AS booked_cons,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS full_call,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) * 1.0000
           / NULLIF(COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END), 0) AS b2l,
       AVG(rehashed * 1.0000) AS app_rehash_rate,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_frac,
       SUM(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_count,
       AVG(active_apr) AS avg_apr
FROM edwnpi.crm_dealer_dim cdd
LEFT JOIN edwnpi.los_deal_current_fact lcdf ON cdd.dealer_number = lcdf.dealer_number
LEFT JOIN sandbox.rehashes rh ON lcdf.loan_id = rh.appid
WHERE cdd.current_version_flag = 1
  AND lcdf.application_received_dtm >= '2025-05-01'
  AND acall_amtfin IS NOT NULL
  AND pricing_hurdle_name IS NOT NULL
  AND pricing_hurdle_name != 'mROA-KMX'
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""
df1 = run_sql(q1)
df1['year'] = df1['year'].astype(int)
df1['month'] = df1['month'].astype(int)
df1['period'] = df1['year'].astype(str) + '-' + df1['month'].astype(str).str.zfill(2)
print(f"Method 1 rows: {len(df1)}")
print(f"\nFull Call Rate by Hurdle and Month (Method 1 - Monthly, All BMS, lcdf):")
pivot1 = df1.pivot_table(index='pricing_hurdle_name', columns='period', values='full_call')
pivot2 = df1.pivot_table(index='pricing_hurdle_name', columns='period', values='low_disc_frac')
print(pivot1.round(4).to_string())
print(pivot2.round(4).to_string())

Method 1 rows: 84

Full Call Rate by Hurdle and Month (Method 1 - Monthly, All BMS, lcdf):
period               2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06
pricing_hurdle_name                                                                                                                              
mROA-AN               0.5580   0.5632   0.5404   0.5781   0.5625   0.5371   0.5542   0.5349   0.5334   0.5155   0.5596   0.5816   0.6138   0.5934
mROA-ENT              0.3951   0.3888   0.3714   0.3918   0.3465   0.3466   0.3410   0.3409   0.3507   0.3272   0.3276   0.2762   0.2653   0.2368
mROA-FLD              0.2992   0.2853   0.2842   0.3310   0.3123   0.3070   0.3086   0.3120   0.3220   0.3325   0.3582   0.3061   0.3107   0.2815
mROA-FRN              0.3653   0.3631   0.3633   0.3987   0.3651   0.3499   0.3523   0.3581   0.3576   0.3558   0.3876   0.3507   0.3414   0.3208
mROA-MCY              0.3832   0.

In [4]:
# Method 7: full_call_tracker_all_weekly - Weekly, all hurdles individually, lcdf-based, since 2025-05-01
# Aggregated to monthly for comparison
q7 = """
SELECT pricing_hurdle_name,
       DATE_PART('year', rh.reqappdate) AS year,
       DATE_PART('week', rh.reqappdate) AS week,
       MIN(rh.reqappdate) AS first_date,
       COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END) AS apps,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) AS cons,
       SUM(booked) AS booked_cons,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS full_call,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN loan_id END) * 1.0000
           / NULLIF(COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN loan_id END), 0) AS b2l,
       AVG(rehashed * 1.0000) AS app_rehash_rate,
       AVG(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_frac,
       SUM(CASE WHEN aspect = 'APPLICATION' THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250 THEN 1.0000 ELSE 0.0000 END END) AS low_disc_count,
       AVG(active_apr) AS avg_apr
FROM edwnpi.crm_dealer_dim cdd
LEFT JOIN edwnpi.los_deal_current_fact lcdf ON cdd.dealer_number = lcdf.dealer_number
LEFT JOIN sandbox.rehashes rh ON lcdf.loan_id = rh.appid
WHERE cdd.current_version_flag = 1
  AND lcdf.application_received_dtm >= '2025-05-01'
  AND acall_amtfin IS NOT NULL
  AND pricing_hurdle_name IS NOT NULL
  AND pricing_hurdle_name != 'mROA-KMX'
GROUP BY 1, 2, 3
HAVING MIN(rh.reqappdate) <= SYSDATE - 5
ORDER BY 1, 2, 3
"""
df7 = run_sql(q7)
df7['first_date'] = pd.to_datetime(df7['first_date'])
df7['period'] = df7['first_date'].dt.to_period('M').astype(str)

# Aggregate weekly to monthly (weighted by apps)
df7_monthly = df7.groupby(['pricing_hurdle_name', 'period']).apply(
    lambda g: pd.Series({
        'full_call': (g['full_call'] * g['apps']).sum() / g['apps'].sum(),
        'apps': g['apps'].sum(),
    })
).reset_index()

print(f"Method 7 rows (weekly): {len(df7)}, aggregated to monthly: {len(df7_monthly)}")
print(f"\nFull Call Rate by Hurdle and Month (Method 7 - Weekly agg to Monthly, All BMS, lcdf):")
pivot7 = df7_monthly.pivot_table(index='pricing_hurdle_name', columns='period', values='full_call')
print(pivot7.round(4).to_string())

Method 7 rows (weekly): 348, aggregated to monthly: 78

Full Call Rate by Hurdle and Month (Method 7 - Weekly agg to Monthly, All BMS, lcdf):
period               2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05
pricing_hurdle_name                                                                                                                     
mROA-AN               0.5593   0.5612   0.5444   0.5746   0.5615   0.5362   0.5527   0.5349   0.5352   0.5159   0.5586   0.5922   0.6100
mROA-ENT              0.3954   0.3877   0.3728   0.3899   0.3452   0.3497   0.3389   0.3409   0.3508   0.3273   0.3230   0.2726   0.2660
mROA-FLD              0.2998   0.2830   0.2893   0.3323   0.3097   0.3103   0.3076   0.3120   0.3235   0.3319   0.3542   0.3033   0.3111
mROA-FRN              0.3660   0.3633   0.3690   0.3953   0.3637   0.3509   0.3505   0.3581   0.3591   0.3565   0.3844   0.3500   0.3378
mROA-MCY              0.3857   0.366

In [5]:
# Side-by-side comparison: Method 1 (monthly grouping) vs Method 7 (weekly agg to monthly)
print(f"{'='*80}")
print(f"  COMPARISON: Full Call Rate by Hurdle x Month")
print(f"  Method 1 = monthly GROUP BY | Method 7 = weekly GROUP BY aggregated to monthly")
print(f"{'='*80}")

comp = df1[['pricing_hurdle_name', 'period', 'full_call', 'apps']].rename(
    columns={'full_call': 'full_call_m1', 'apps': 'apps_m1'}
).merge(
    df7_monthly[['pricing_hurdle_name', 'period', 'full_call', 'apps']].rename(
        columns={'full_call': 'full_call_m7', 'apps': 'apps_m7'}
    ),
    on=['pricing_hurdle_name', 'period'],
    how='outer'
).sort_values(['pricing_hurdle_name', 'period']).reset_index(drop=True)

comp['diff'] = comp['full_call_m7'] - comp['full_call_m1']
comp['diff_pct'] = comp['diff'] / comp['full_call_m1'] * 100

print(f"\nOverall avg full_call:")
print(f"  Method 1 (monthly):   {comp['full_call_m1'].mean():.4f}")
print(f"  Method 7 (weekly agg): {comp['full_call_m7'].mean():.4f}")
print(f"  Avg difference:        {comp['diff'].mean():.4f} ({comp['diff_pct'].mean():.2f}%)")

print(f"\nBy Hurdle:")
hurdle_comp = comp.groupby('pricing_hurdle_name').agg(
    full_call_m1=('full_call_m1', 'mean'),
    full_call_m7=('full_call_m7', 'mean'),
    diff=('diff', 'mean'),
    diff_pct=('diff_pct', 'mean')
).round(4)
print(hurdle_comp.to_string())

print(f"\nDetail:")
comp.round(4)

  COMPARISON: Full Call Rate by Hurdle x Month
  Method 1 = monthly GROUP BY | Method 7 = weekly GROUP BY aggregated to monthly

Overall avg full_call:
  Method 1 (monthly):   0.3968
  Method 7 (weekly agg): 0.3989
  Avg difference:        -0.0002 (-0.05%)

By Hurdle:
                     full_call_m1  full_call_m7    diff  diff_pct
pricing_hurdle_name                                              
mROA-AN                    0.5590        0.5567  0.0003    0.0644
mROA-ENT                   0.3361        0.3431 -0.0007   -0.2118
mROA-FLD                   0.3108        0.3129 -0.0001   -0.0157
mROA-FRN                   0.3593        0.3619 -0.0003   -0.0853
mROA-MCY                   0.3357        0.3385 -0.0002   -0.0403
mROA-STG                   0.4797        0.4802 -0.0002   -0.0315

Detail:


,pricing_hurdle_name,period,full_call_m1,apps_m1,full_call_m7,apps_m7,diff,diff_pct
0,mROA-AN,2025-05,0.5580,12227,0.5593,12387.0,0.0013,0.2305
1,mROA-AN,2025-06,0.5632,10751,0.5612,12571.0,-0.0020,-0.3619
2,mROA-AN,2025-07,0.5404,11681,0.5444,10747.0,0.0040,0.7347
3,mROA-AN,2025-08,0.5781,11915,0.5746,10869.0,-0.0035,-0.6033
4,mROA-AN,2025-09,0.5625,10275,0.5615,11965.0,-0.0010,-0.1761
5,mROA-AN,2025-10,0.5371,10285,0.5362,9105.0,-0.0009,-0.1615
6,mROA-AN,2025-11,0.5542,10326,0.5527,9816.0,-0.0015,-0.2619
7,mROA-AN,2025-12,0.5349,9627,0.5349,9627.0,-0.0000,-0.0026
8,mROA-AN,2026-01,0.5334,9436,0.5352,9594.0,0.0018,0.3365
9,mROA-AN,2026-02,0.5155,14301,0.5159,14420.0,0.0004,0.0733
